# LifeLog PySpark Analysis

This notebook provides PySpark examples for querying processed CSV files.

In [7]:
# Import PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc, avg, sum as spark_sum
import pyspark.sql.functions as F
import findspark

# Initialize findspark (helps locate Spark installation)
findspark.init()

# At the top of your notebook, increase max rows displayed
from IPython.core.display import display, HTML
display(HTML("<style>.output_result { max-height:10000px !important; }</style>"))

In [2]:
# Force full output
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [3]:
# Create SparkSession
spark = SparkSession.builder \
    .appName("LifeLog Analysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark running on: {spark.sparkContext.master}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/01 13:37:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.0
Spark running on: local[*]


In [4]:
# Load music data as example
music_df = (
  spark.read
       .option("delimiter", "|")
       .option("header", "true")
       .option("inferSchema", "true")
       .csv("../files/website_files/music/music_page_data.csv")
)

In [10]:
(music_df.withColumn("rounded_completion", F.round(col("completion"), 1))
         .filter(col("rounded_completion") == "0.0").select("listening_seconds").distinct().show()

+-----------------+
|listening_seconds|
+-----------------+
|               12|
|                1|
|               13|
|                6|
|                3|
|                5|
|               15|
|                9|
|               17|
|                4|
|                8|
|                7|
|               10|
|               11|
|               14|
|                2|
|                0|
|               18|
|               22|
|               16|
+-----------------+
only showing top 20 rows



In [11]:
# Show schema
total_count = music_df.count()
(music_df.withColumn("rounded_completion", F.round(col("completion"), 2))
         .groupBy("rounded_completion")
         .count()
         .withColumn("%_count", F.col("count") / total_count)
         .orderBy("rounded_completion")
).show()

+------------------+------+--------------------+
|rounded_completion| count|             %_count|
+------------------+------+--------------------+
|               0.0|101210|   0.333522266672818|
|              0.01| 32340| 0.10657158486512137|
|              0.02|  8562|0.028214777662806716|
|              0.03|  3759|0.012387216682374497|
|              0.04|  2357|0.007767137462185871|
|              0.05|  1653|0.005447211805258...|
|              0.06|  1307|0.004307021070461151|
|              0.07|  1115|0.003674314073117...|
|              0.08|   961|0.003166830335664...|
|              0.09|   865|0.002850476836992269|
|               0.1|   796|0.002623097759821...|
|              0.11|   791|0.002606621015099289|
|              0.12|   650|0.002141976813924...|
|              0.13|   683|0.002250723329093...|
|              0.14|   664|0.002188111699147823|
|              0.15|   647|0.002132090767091327|
|              0.16|   611|0.002013458205089337|
|              0.17|

26/01/01 15:16:45 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 987122 ms exceeds timeout 120000 ms
26/01/01 15:16:45 WARN SparkContext: Killing executors is not supported by current scheduler.
26/01/01 15:32:46 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o